Created 8/11/2022

Trying to analyze how many times the model makes swap errors

Goal: When the model responds incorrectly (i.e. large error?) -->

plot absolute(target-output) vs. absolute(output-other stimuli in memory(?)) - but that is only helpful in the 2 task case.; output gating problem 
or could do 
absolute(target-output) vs absolute(output-other stimuli) - this is what should have been remembered - i.e. input gating problem; this is only helpful in the 2 task case.  


could do: 

plot absolute(target-output) vs. min(absolute(output - other stimuli in memory)) 


OR could when absolute(target-output)>10, count swap error if items in stripes are >10 away from the target and <10 away from the response.


looks like we will need the stripes from previous trial - because it gets removed from stripe before end of the trial. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import random
sns.set()
from decodedoutputdefs import Precision, training_curve_indv, training_curve_average #class with the compiled definitions

In [2]:
base_file ="Y:/sir2_new/results/Ring/2Stripe/decodedrew/contNoStimLoc/NoIgnore/" 
file = 'sir2model0_RewThreshold5.5_decoded_Base.csv'

In [3]:
df = pd.read_csv(base_file+file, sep = '\t')

In [4]:
p = Precision()
err = p.mult_models([base_file+file])

In [6]:
df = p.df_recall
df_full = p.dat
stripes = [i for i in df.keys() if 'stripe' in i]

In [51]:
prev_trial = df_full[df_full['|Trial']==(trial-1)]

In [52]:
prev_trial['#stripe'][0]

242.6

In [117]:

def find_swap_error(df,df_full,error_dist = 15):
    swap_error = []
    for i in range(len(err)):
        if err[i] > error_dist: #not the correct response. need to modulate the error_dist for chunking purposes
            output = df.iloc[i]['#OutDecode']
            target = df.iloc[i]['#OutTarget']

            trial = df.iloc[i]['|Trial'] # need to get the trial - 1  for the stripes.
            if trial != 0:
                prev_trial = df_full[df_full['|Trial']==(trial-1)]
                stripe1 = prev_trial['#stripe'][0]
                stripe2 = prev_trial['#stripe\x01'][0]

#                 if np.logical_or(stripe1-output<error_dist, stripe2-output<error_dist): #stripe1-output<error_dist checks if the response was from stripe1 or from stripe2.
#                     swap_error.append(trial)
                if np.logical_and(stripe1-output<error_dist, stripe1!=0):
                    swap_error.append(trial)
                if np.logical_and(stripe2-output<error_dist, stripe2!=0):
                    swap_error.append(trial)
    return swap_error
        

In [145]:
#since the trial numbers are NOT unique - need to go through each model separately. 
base_file = 'Y:/sir4/results/Ring/2Stripe/decodedrew/contNoStimLoc/NoIgnore/'
files = os.listdir(base_file)
files = [f for f in files if '.csv' in f]
full_files = [base_file+f for f in files if 'EpcLog' not in f]
swap_errors = []
total_trials = []
for f in full_files:
    p = Precision()
    err = p.mult_models([f])
    total_trials.append(len(err))
    swap_error = find_swap_error(p.df_recall,p.dat, error_dist = 15)
    swap_errors.append(len(swap_error)) #recording the number of swap errors made in each model (not the trial numbers anymore.)
sir4 = sum(swap_errors)/sum(total_trials)

In [146]:
print(sir2_new)
print(sir3)
print(sir4)

0.04742489270386266
0.20474554434185097
0.19362806515124395


In [141]:
print(sir2_chunk)
print(sir3_chunk)
print(sir4_chunk)

0.03129918443267993
0.24895315128306073
0.23041689032027196


In [148]:
#lets get an example of this 
df = p.df_recall 
df_full = p.dat
df[df['|Trial']==42]

#df_full[df_full['|Trial']==42]

,|Run,|Epoch,|Trial,$TrialName,#Err,#TrlDecodedDiff,#SSE,#AvgSSE,#CosDiff,#DA,...,"#PFCoutD[4:1,0,0,18]","#PFCoutD[4:1,0,0,19]",#stripe,#stripe,#OutDecode,#OutTarget,#InDecode,$Lesion,#LesionProp,$LesionApplied
True,0,-1,42,Recall1_342.6209021533586_mnt1_-1_mnt2_-1_mnt3...,1,260.2,4.952,0.2476,-0.3368,-0.8499,...,1.401000e-45,1.401000e-45,0.0,85.13,84.93,345.1,0.0,StimLoc,0,no


In [151]:
df_full[df_full['|Trial']==40]['$TrialName'][0]

'Recall4_104.77838829146755_mnt1_342.6209021533586_mnt2_-1_mnt3_85.19893985448522_mnt4_-1_rew_0.890447277492947'

In [115]:
df[df['|Trial']==6]

,|Run,|Epoch,|Trial,$TrialName,#Err,#TrlDecodedDiff,#SSE,#AvgSSE,#CosDiff,#DA,...,"#PFCoutD[4:1,0,0,19]",#stripe,#stripe,#ChunkDecode,#OutDecode,#OutTarget,#InDecode,$Lesion,#LesionProp,$LesionApplied
True,0,-1,6,Recall3_331.0666533422646_mnt1_58.624765526467...,1,335.1,3.517,0.1759,-0.2579,-1.252,...,1.401000e-45,0.0,0,0.0,0.0,335.1,0.0,StimLoc,0,no


In [155]:
f = 'Y:/sir4_chunk/results/Ring/2Stripe/hybrid/decodedrew/contNoStimLoc/NoIgnore/sir4chunkhybridmodel9_RewThreshold5.5_decoded_Base.csv'
p = Precision()
err = p.mult_models([f])
total_trials.append(len(err))
swap_error = find_swap_error(p.df_recall,p.dat, error_dist = 15)

In [157]:
df_full = p.dat
df = p.df_recall

In [160]:
df_full[np.logical_and(df_full['|Trial']<25,df_full['|Trial']>18)]

,|Run,|Epoch,|Trial,$TrialName,#Err,#TrlDecodedDiff,#SSE,#AvgSSE,#CosDiff,#DA,...,"#PFCoutD[4:1,0,0,19]",#stripe,#stripe,#ChunkDecode,#OutDecode,#OutTarget,#InDecode,$Lesion,#LesionProp,$LesionApplied
True,0,-1,19,Recall1_349.4358362408166_mnt1_-1_mnt2_-1_mnt3...,1,351.400,3.857,0.19280,-0.5180,-1.0920,...,1.401000e-45,223.1,0,0.0,0.0,351.4,0.0,StimLoc,0,no
False,0,-1,20,Store4_260.4622223395245_mnt1_-1_mnt2_-1_mnt3_...,0,11.510,0.000,0.00000,0.8732,0.0000,...,1.401000e-45,255.0,0,258.0,249.7,261.2,261.1,StimLoc,0,no
True,0,-1,21,Recall3_223.9603948137716_mnt1_-1_mnt2_-1_mnt3...,1,31.280,2.192,0.10960,0.4494,0.2922,...,1.401000e-45,255.6,0,0.0,255.7,224.5,0.0,StimLoc,0,no
False,0,-1,22,Store2_225.58885313661816_mnt1_-1_mnt2_225.588...,0,4.683,0.000,0.00000,0.9413,0.0000,...,1.401000e-45,232.7,0,227.9,230.5,225.9,225.8,StimLoc,0,no
False,0,-1,23,Store1_232.45267335320264_mnt1_232.45267335320...,0,1.141,0.000,0.00000,0.9846,0.0000,...,1.401000e-45,229.6,0,233.4,232.8,231.7,231.7,StimLoc,0,no
True,0,-1,24,Recall2_225.58885313661816_mnt1_232.4526733532...,1,10.740,1.565,0.07827,0.6568,0.5888,...,1.401000e-45,229.6,0,0.0,236.6,225.9,0.0,StimLoc,0,no


In [156]:
swap_error

[21,
 47,
 53,
 78,
 86,
 88,
 109,
 111,
 122,
 130,
 139,
 141,
 143,
 153,
 156,
 169,
 171,
 180,
 218,
 238,
 249,
 276,
 279,
 286,
 292,
 307,
 314,
 331,
 338,
 376,
 386,
 408,
 438,
 452,
 455,
 463,
 475,
 487,
 490,
 505,
 507,
 527,
 532,
 537,
 550,
 573,
 591,
 645,
 666,
 673,
 681,
 692]